In [1]:
!pip install geopandas folium scipy pandas numpy

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/durlabhjilegend-lgtm/delhi-aqi-dataset/main/aqi_data.csv"

df = pd.read_csv(url)

df["timestamp"] = pd.to_datetime(df["timestamp"])

df.head()

,station,latitude,longitude,aqi,timestamp
0,"DITE Okhla, Delhi, Delhi, India",28.531314,77.270686,209,2026-03-07 16:07:26.633441
1,"Sector - 125, Noida, India",28.544761,77.323126,349,2026-03-07 16:07:26.633445
2,"Anand Vihar, Delhi, Delhi, India",28.650800,77.315200,175,2026-03-07 16:07:26.633446
3,"Lodhi Road, Delhi, Delhi, India",28.589701,77.221917,129,2026-03-07 16:07:26.633446
4,"Bramprakash Ayurvedic Hospital, Najafgarh, Del...",28.572714,76.933433,109,2026-03-07 16:07:26.633447


In [4]:
latest = df.sort_values("timestamp").groupby("station").tail(1)
print(latest)

                                                station   latitude  longitude  \
7238           Knowledge Park - V, Greater Noida, India  28.557054  77.453663   
7239                         Teri Gram, Gurugram, India  28.427500  77.146500   
7241                             Loni, Ghaziabad, India  28.757294  77.278792   
7240            Sri Auribindo Marg, Delhi, Delhi, India  28.528344  77.189304   
7242           Pooth Khurd, Bawana, Delhi, Delhi, India  28.775796  77.046251   
7243                           Sector-116, Noida, India  28.569230  77.393848   
7244              ITI Jahangirpuri, Delhi, Delhi, India  28.733016  77.171970   
7245               Burari Crossing, Delhi, Delhi, India  28.725646  77.203384   
7246                   Anand Vihar, Delhi, Delhi, India  28.650800  77.315200   
7247                          Pusa, Delhi, Delhi, India  28.636997  77.172248   
7248                  Punjabi Bagh, Delhi, Delhi, India  28.668300  77.116700   
7249                        

In [5]:
import geopandas as gpd

stations = gpd.GeoDataFrame(
    latest,
    geometry=gpd.points_from_xy(latest.longitude, latest.latitude),
    crs="EPSG:4326"
)

In [6]:
import folium

m = folium.Map(location=[28.61,77.23], zoom_start=11, tiles="cartodbpositron")

for _,row in stations.iterrows():

    folium.CircleMarker(
        location=[row["latitude"],row["longitude"]],
        radius=6,
        color="blue",
        fill=True,
        popup=f"{row['station']} | AQI {row['aqi']}"
    ).add_to(m)

m

In [8]:
stations.columns

Index(['station', 'latitude', 'longitude', 'aqi', 'timestamp', 'geometry'], dtype='object')

In [9]:
stations["aqi"] = pd.to_numeric(stations["aqi"], errors="coerce")

In [10]:
stations = stations.dropna(subset=["aqi"])

In [11]:
import numpy as np
from scipy.spatial import cKDTree

coords = stations[["longitude","latitude"]].to_numpy()

values = stations["aqi"].to_numpy()

tree = cKDTree(coords)

In [12]:
df = df[df["aqi"] != "-"]

In [13]:
import numpy as np
from scipy.spatial import cKDTree

coords = stations[["longitude","latitude"]].to_numpy()
values = stations["aqi"].astype(float).to_numpy()

tree = cKDTree(coords)

def idw(lon,lat,k=6):

    dist,idx = tree.query([lon,lat],k=k)

    weights = 1/(dist**2 + 1e-9)

    return np.sum(weights*values[idx])/np.sum(weights)

In [14]:
from shapely.geometry import Point

xmin,ymin,xmax,ymax = stations.total_bounds

grid_points = []

for x in np.arange(xmin,xmax,0.01):
    for y in np.arange(ymin,ymax,0.01):

        grid_points.append(Point(x,y))

grid = gpd.GeoDataFrame(geometry=grid_points, crs="EPSG:4326")

In [15]:
grid["aqi"] = grid.geometry.apply(lambda p: idw(p.x,p.y))

In [16]:
from folium.plugins import HeatMap

heat_data = [[p.y,p.x,aqi] for p,aqi in zip(grid.geometry,grid["aqi"])]

HeatMap(heat_data,radius=20,blur=15).add_to(m)

m

In [17]:
wards = gpd.read_file(
"https://raw.githubusercontent.com/durlabhjilegend-lgtm/delhi-aqi-dataset/main/delhi_wards.geojson"
)

In [18]:
grid = gpd.sjoin(grid,wards[["WardName","geometry"]],predicate="within")

ward_aqi = grid.groupby("WardName")["aqi"].mean().reset_index()

wards = wards.merge(ward_aqi,on="WardName")

In [19]:
def citizen_alert(aqi):

    if aqi > 300:
        return "Severe pollution: avoid outdoor activity and wear masks."

    elif aqi > 200:
        return "Very poor air: limit outdoor exposure."

    elif aqi > 150:
        return "Poor air quality: sensitive groups should take precautions."

    else:
        return "Air quality acceptable."

In [20]:
def govt_action(aqi):

    if aqi > 300:
        return "Deploy emergency mitigation teams and identify pollution sources."

    elif aqi > 200:
        return "Investigate pollution sources and enforce dust control."

    elif aqi > 150:
        return "Increase monitoring and enforce construction regulations."

    else:
        return "No immediate action required."

In [21]:
wards["citizen_alert"] = wards["aqi"].apply(citizen_alert)

wards["govt_action"] = wards["aqi"].apply(govt_action)

In [22]:
priority = wards.sort_values("aqi",ascending=False)[
    ["WardName","aqi","govt_action"]
]

priority.head(10)

,WardName,aqi,govt_action
36,PAHAR GANJ,157.023022,Increase monitoring and enforce construction r...
228,DELHI GATE,153.721279,Increase monitoring and enforce construction r...
227,CHANDANI MAHAL,151.932199,Increase monitoring and enforce construction r...
152,RAM NAGAR,149.750079,No immediate action required.
11,NARAINA,149.405502,No immediate action required.
39,WEST PATEL NAGAR,147.772429,No immediate action required.
226,JAMA MASJID,146.864928,No immediate action required.
41,KARAM PURA,145.598090,No immediate action required.
150,BALLIMARAN,144.864498,No immediate action required.
42,MOTI NAGAR,144.112414,No immediate action required.
